In [ ]:
# %pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130
# %pip install scikit-learn
# %pip install torch.utils
# %pip install pandas matplotlib

# PyTorch Regression with Sklearn California Housing
## Goal

Build a regression model using `PyTorch` to predict a target variable from the California Housing Dataset. 
This will integrate `pandas` for data handling and `scikit-learn` for preprocessing before feeding the data into a PyTorch neural network.

## Tasks

1. **Data Fetching:** Fetch the diabetes dataset from sklearn and convert it to a `pandas DataFrame`.

2. **Data Preprocessing:** 

    * Separate features (X) and target (y).
    * Split the dataset into training and testing sets using train_test_split.
    * Scale features (X) using StandardScaler.

3. **PyTorch Data Preparation:**

    * Convert preprocessed NumPy Arrays into Tensor objects.
    * Create custom Datatset class to handle data.
    * Create DataLoader for training and testing datasets.

4. **Define Neural Network:** Complete `__init__` and `forward` methods of the `HousingModel` class. 
    Use `nn.Linear` layers, and adjust input features for the preprocessed data.

5. **Define Loss Function and Optimizer:** Choose and create appropriate objects for regression.

6. **Training Loop:** Write the training loop for ### epochs, iterating through the `DataLoader`.

7. **Evaluate and Vizualize:** Assess model performance on the test set and visualize training loss and predictions


# 1. Import Libraries

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available else "cpu")
print(f"Using device {device}")

# 2. Data Loading
Load the `california_housing` dataset from `sklearn.datasets` and convert it to a `pandas.DataFrame`. 
Display general information

In [ ]:
# Fetches california housing directly as a Pandas DataFrame
housing_df = fetch_california_housing(as_frame = True).frame

In [ ]:
# Display first 5 rows of housing data
print(housing_df.head())
print()

# Display general information of housing data
print(housing_df.info())
print()

# Display statistical information of housing data
print(housing_df.describe())
print()

# Display size of the housing data
print(housing_df.size)

## 3. Data Preprocessing with Pandas and Scikit-learn
**Do:**
1. Separate the features (X) and the target (y) from the DF. 
2. Split the data into training and testing sets using tts. 
3. Apply `StandardScaler` to the training and testing features. This improves neural network performance

In [ ]:
# Set features DF and target series
X = housing_df[housing_df.columns[:-1]]
y = housing_df.loc[:, "MedHouseVal"]



In [ ]:
# Split using tts with 80/20 and 42 random state
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.20, random_state = 42)

# Print shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Create scaler object
scaler = StandardScaler()

print("Features before scaling: ")
print(X_train.head())

# Scale features
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Scale targets
y_train_scaled = scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler.transform(y_test.values.reshape(-1, 1))

print("Features after scaling: ")
print(X_train_scaled[:5])
print()
print(y_train_scaled)

## 4. PyTorch Data Preparatioin
**Do:**
1. Convert preprocessed and scaled NumPy arrays into Tensor objects
2. Create a custom dataset class to handle the data
3. Create a DataLoader for training and testing the datasets

In [ ]:
# Checking shapes
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

# Because the target values are one-dimensional, we need to unsqueeze them to increase their dimensions without impacting the data

In [ ]:
# Feature tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32)

# Target tensors
y_train_tensor = torch.tensor(y_train_scaled, dtype = torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype = torch.float32)

In [ ]:
print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"y_train_tensor shape: {y_train_tensor.shape}")

# Because we unsqueezed the target values, our target dimension went from 1-d (n, ) => 2-d (n, n)

In [ ]:
class HousingDataset(Dataset): 
    def __init__(self, features, target): 
        self.features = features
        self.target = target

    def __len__(self):
        return len(self.features)
    
    def __getitem__(self, idx): 
        return self.features[idx], self.target[idx]

In [ ]:
# Create our dataset objects
train_dataset = HousingDataset(X_train_tensor, y_train_tensor)
test_dataset = HousingDataset(X_test_tensor, y_test_tensor)

In [ ]:
print(len(train_dataset))
print(train_dataset[:5])

print()

print(len(test_dataset))
print(test_dataset[:5])

In [ ]:
train_loader = DataLoader(
    dataset = train_dataset, 
    batch_size = 32, 
    shuffle = True, 
    #pin_memory = True,
    #num_workers = 2
)

test_loader = DataLoader(
    dataset = test_dataset, 
    batch_size = 32, 
    shuffle = False
)

In [ ]:
print("First batch from train_loader")

for batch_X, batch_y in train_loader: 
    print(f"Batch_X shape: {batch_X.shape} \nBatch_y shape: {batch_y.shape}")
    break

# 5. Define Neural Network
**Do:**
1. Create `HousingModel` class
2. Complete `__init__` and `forward` methods of the model
3. Use nn.Linear layers 

In [ ]:
class HousingModel(nn.Module): 
    def __init__(self, input_features): 
        super(HousingModel, self).__init__()

        self.network = nn.Sequential(
            nn.Linear(input_features, 128), 
            nn.ReLU(), 
            nn.Dropout(p = 0.2),

            nn.Linear(128, 64),
            nn.ReLU(), 
            nn.Dropout(p = 0.1),

            nn.Linear(64, 32), 
            nn.ReLU(), 
            
            nn.Linear(32, 1)
        )

    def forward(self, x): 
        return self.network(x)
    


In [ ]:
# Dimension count to be passed into model
input_dimension = X_train_tensor.shape[1]

# Creating the model with the number of input features as an argument
model = HousingModel(input_dimension)

print("Model Architecture")
print(model)


## 6. Define Loss Function and Optimizer
**Do:**
1. Choose and create Loss Function Object (MSE, L1, Huber, etc.)
2. Choose and create Optimizer object (Adam or sgd)

In [ ]:
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.0005, weight_decay = 1e-4)

## 7. Training Loop
**Do:** 
1. Define # of epochs to iterate through
2. Write training loop for # of epochs iterating through DataLoader

In [ ]:
epochs = 300
losses = []

for epoch in range(epochs): 
    epoch_loss = 0.0
    
    for batch_X, batch_y in train_loader: 
        # Zero the gradient
        optimizer.zero_grad()
        # Make prediction
        y_pred = model(batch_X)
        # Calculate loss
        loss = loss_fn(y_pred, batch_y)
        # Create gradient
        loss.backward()
        # Adjust weights
        optimizer.step()
        # Add losses to epoch loss: 
        epoch_loss += loss.item()

    losses.append(epoch_loss)

    if (epoch + 1) % 20 == 0: 
        print(f"Epoch [{epoch + 1} / {epochs}], Loss: {losses[-1]:.4f}")

print("Training Complete")

## 8. Evaluate & Visualize
**Do:**
1. Plot model test set and visualize training loss/predictions
2. Assess model performance

In [ ]:
plt.figure(figsize = (8, 4))
plt.plot(losses)
plt.ylabel("Loss")
plt.xlabel("Epoch")
plt.title("Training Loss over Epochs")
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
model.eval()

test_predictions = []
test_true_values = []

with torch.no_grad(): 
    for batch_X, batch_y in test_loader: 
        outputs = model(batch_X)
        
        preds = outputs.numpy().reshape(-1)
        labels = batch_y.numpy().reshape(-1)

        test_predictions.extend(preds)
        test_true_values.extend(labels)
    
test_predictions = np.array(test_predictions).flatten()
test_true_values = np.array(test_true_values).flatten()

test_mse = mean_squared_error(test_true_values, test_predictions)
test_r2 = r2_score(test_true_values, test_predictions)

print(f"MSE: {test_mse:.4f}")
print(f"R2: {test_r2:.4f}")

In [ ]:
print(test_predictions.shape)
print(test_true_values.shape)

In [ ]:
print("Preds | Actual")
for i in range(5):
    print(f"{test_predictions[i]:.2f} | {test_true_values[i]:.2f}")

# Check the range
print(f"\nPred Range: {test_predictions.min():.2f} to {test_predictions.max():.2f}")
print(f"True Range: {test_true_values.min():.2f} to {test_true_values.max():.2f}")

In [ ]:
print(f"Pred Mean: {test_predictions.mean():.4f}")
print(f"True Mean: {test_true_values.mean():.4f}")

In [ ]:
plt.figure(figsize = (10, 6))
plt.scatter(test_true_values, test_predictions, alpha = 0.6)
plt.plot([test_true_values.min(), test_true_values.max()], [test_true_values.min(), test_true_values.max()], 'r--')
plt.xlabel("True Test Values")
plt.ylabel("Predicted Test Values")
plt.title("Test Set: True vs. Predicted Values")
plt.grid(True)
plt.show()

In [ ]:
for name, param in model.named_parameters():
  if param.requires_grad:
    print(f"{name}: {param.data.numpy()}")